In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
#to read the dataset, insert file path in the middle
df = pd.read_csv(f"/kaggle/input/q1-ka-ai-2026/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
#

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

df.columns.drop("Order_ID")

In [ ]:
# Task 2: Write your code here:
# print("\nMissing values:")
# print(df.isnull().sum())

#Drop rows with ANY missing values
# df_dropna = df.dropna()
# print("\n After dropna():", len(df_dropna), "rows")


def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  #this is to focus on looking at the columns with missing data instead o printing all columns
  print(missing_values[missing_values > 0])

  if missing_values.any():
    #print("\n I will handle missing values by dropping rows with ANY missing values.")
    df_dropna = df.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])
    print("\n After dropna():", len(df_dropna), "rows")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
categories = pd.DataFrame({"Categorical": ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']}) # DataFrame of one column

print('data before encoding:\n', categories) #show before encoding

onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder (make an object),, this means you get a regular array, not a compressed one
data_onehot_encoded = onehot_encoder.fit_transform(categories) # Apply fit_transform to the copied (learns what categories exist and converts them to binary columns (0s and 1s)

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding


In [ ]:
# Task 5: Write your code here:
# Do we have different scales in the data?
df.describe()

#Rule of thumb: If features have vastly different ranges, then scale them.
#Scaling won't hurt, and it's recommended regardless.
# use standard scaler

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
#NOTE I feel like this cell should be kept empty
import seaborn as sns
# Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Import models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score


# Task 1: Write your code here:
# We need to split our data into X (features) and y (target).

#X = df.drop("Delivery_Time", axis=1).astype(str)
#y = df['Delivery_Time'].astype(float)

X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Legendary in train: {y_train.sum()}, in test: {y_test.sum()}")

# 3. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np

# Sample data
X = np.random.rand(100, 5)
y = np.random.randint(0, 2, 100)

# KFold Cross-Validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression()

# Get scores for each fold
scores = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
print("KFold scores:", scores)
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

# 4. Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# another option Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
print("Random Forest R²:", rf.score(X_test, y_test))

# 1. Mean Absolute Error (MAE)
mae = mean_absolute_error(y_true, y_pred)
print(f"MAE: {mae:.2f}")
print("  → Average absolute error")
print("  → Easy to interpret (same units as target)")

# 6. Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


In [ ]:
# Task 1: Write your code here:


# Gather importances from the models (from the last fold)
print("\nTop 3 important features (Random Forest):")
importances = rf.feature_importances_
for i in np.argsort(importances)[-3:][::-1]:
    print(f"  Feature {i}: {importances[i]:.3f}")

#also another option
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

# 5. Predict
y_pred = model.predict(X_test_scaled)

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: